In [ ]:
# create a document 

from langchain_core.documents import Document

In [ ]:
doc = Document(
    page_content = "the is the data for creating rag",
    metadata={
        "source": "internet.com",
        "pages": 23,
        "author" : "ayush", 
        "date_created": "2025-03-22"
    }
)
doc

In [ ]:
# create a txt file
# import os
# os.makedirs("../data/text_file",exist_ok=False)

In [ ]:
# add data in txt file
sample_text = {
    "../data/text_file/training.txt" : """
    A program is many things. It is a piece of text typed by a programmer, it is
the directing force that makes the computer do what it does, it is data in the
computer's memory, and, at the same time, it controls the actions performed
on this memory. Analogies that try to compare programs to familiar objects
tend to fall short. A superficially fitting one is to compare a program to a
machine—lots of separate parts tend to be involved, and to make the whole
thing tick, we have to consider the ways in which these parts interconnect and
contribute to the operation of the whole.
A computer is a physical machine that acts as a host for these immaterial
machines. Computers themselves can do only stupidly straightforward things.
The reason they are so useful is that they do these things at an incredibly
high speed. A program can ingeniously combine an enormous number of these
simple actions to do very complicated things.
A program is a building of thought. It is costless to build, it is weightless,
and it grows easily under our typing hands. But as a program grows, so does
its complexity. The skill of programming is the skill of building programs that
don't confuse the programmer. The best programs are those that manage to
do something interesting while still being easy to understand.
    """
}

for filepath, content in sample_text.items():
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(content)


print("sample data data created!")

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_file/training.txt", encoding="utf-8")
document = loader.load()
print(document)

In [ ]:
#  load data from a file

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls = PyMuPDFLoader)

pdf_document = dir_loader.load()
print(pdf_document)

In [ ]:
# data chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter;

documents = dir_loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(len(chunks))
print(chunks[1])

In [ ]:
# vector embedding and database
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import *
from sklearn.metrics.pairwise import cosine_similarity
import os


In [ ]:
# generate embedding

class EmbeddingManager:
    def __init__(self, model_name:str = 'all-MiniLm-L6-v2'):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self) :
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded sucessfully, model dimention: {self.model}")
        except Exception as e:
            print(f"error while loading model {self.model_name}: {e}")
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded")
        

        print(f"generating embedding for {len(texts)} texts")
        embedding = self.model.encode(texts, show_progress_bar = True)
        print(f"generated embedding with shape: {embedding.shape}")
        return embedding
    
embedding_manager=EmbeddingManager()
embedding_manager

In [ ]:
#  vector store
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory="../data/vector_store"):
        self.connection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "pdf document embedding for rag"}
            )
            print(f"vector store initialized, collection: {self.collection_name}")
            print(f"existing document in collection: {self.collection_count()}")

        except Exception as e:
            print(f"error initializing vector store: {e}")
            raise


        def add_document(self, documents: List[any], embeddings: np.ndarray):
            if len(documents) != len(embeddings):
                raise ValueError("number of docs must match no of embeddings")
        print(f"adding {len(documents)} document to vector store")

        # data for db store

        ids=[]
        metadata=[]
        documents_text=[]
        embedding_list=[]

        # generate unique id

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
